# Notebook 02 - Journey-Aware Verification Demo (Executable)

**NeurIPS 2026 Workshop - Contribution 2: Journey-Aware Verification**

This notebook demonstrates a deterministic, framework-agnostic comparison of two aggregation methods using the **same mocked turn-level quality scores**:

1. `Simple average` of turn scores
2. `Journey-aware weighted aggregation` based on business-critical stage importance

## Core argument

A simple average can hide failures in business-critical stages.
Journey-aware aggregation reduces that masking effect by assigning larger influence to high-risk stages, especially:

- `eligibility_or_suitability`
- `recommendation_or_guidance`

## What this notebook does

1. Loads synthetic conversation scenarios (no credentials, no LLM calls).
2. Displays turn-level conversation data with expected stage and mocked score.
3. Optionally runs rule-based journey classification for transparency.
4. Compares simple average vs journey-aware weighted score per scenario.
5. Highlights downgrade cases where business-critical failures are diluted by unweighted averaging.

This notebook **does not** introduce a new scoring metric. It changes only aggregation logic.

In [ ]:
# Cell 1 - Imports and setup
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from workshop_verify_agents.examples.conversations import list_scenarios, get_scenario
from workshop_verify_agents.journey import (
    STAGE_REGISTRY,
    classify_conversation,
    compare_conversation,
    stage_weights,
    stage_weight,
    simple_average_score,
    journey_weighted_score,
)

REPO_ROOT = Path("../../")
EXAMPLES_DIR = REPO_ROOT / "src/workshop_verify_agents/examples"

print("Journey-aware demo imports loaded.")
print("Total taxonomy stages:", len(STAGE_REGISTRY))
print("Default stage weights:", stage_weights())

In [ ]:
# Cell 2 - Load synthetic scenarios and optional JSON cross-check
scenario_names = list_scenarios()
scenarios = [get_scenario(name) for name in scenario_names]

display(Markdown(f"### Loaded scenarios from examples/conversations.py: {len(scenarios)}"))
display(pd.DataFrame(
    [
        {
            "scenario_name": s["scenario_name"],
            "turn_count": len(s["turns"]),
            "expected_stage_count": len(s["expected_stage_labels"]),
            "mocked_score_count": len(s["mocked_quality_scores"]),
        }
        for s in scenarios
    ]
))

# Optional consistency check with synthetic_journeys.json
json_path = EXAMPLES_DIR / "synthetic_journeys.json"
if json_path.exists():
    with open(json_path, encoding="utf-8") as f:
        journeys_json = json.load(f)
    print("JSON scenarios available:", len(journeys_json.get("journeys", [])))
else:
    print("synthetic_journeys.json not found; proceeding with Python scenarios only.")

In [ ]:
# Cell 3 - Conversation tables with expected stages, mocked scores, and weights
scenario_tables = {}
classifier_tables = {}

for scenario in scenarios:
    scenario_name = scenario["scenario_name"]
    turns = scenario["turns"]
    expected_stages = scenario["expected_stage_labels"]
    mocked_scores = scenario["mocked_quality_scores"]

    rows = []
    for idx, (turn, stage_name, score) in enumerate(zip(turns, expected_stages, mocked_scores)):
        rows.append({
            "turn_index": idx,
            "role": turn["role"],
            "text": turn["text"],
            "expected_stage": stage_name,
            "mocked_quality_score": score,
            "stage_weight": stage_weight(stage_name),
        })
    conversation_df = pd.DataFrame(rows, columns=[
        "turn_index",
        "role",
        "text",
        "expected_stage",
        "mocked_quality_score",
        "stage_weight",
    ])
    scenario_tables[scenario_name] = conversation_df

    # Optional deterministic classifier output for transparency
    classified = classify_conversation(turns)
    classifier_tables[scenario_name] = pd.DataFrame(
        [
            {
                "turn_index": item.turn_index,
                "classified_stage": item.stage.value,
                "confidence": item.confidence,
                "rationale": item.rationale,
            }
            for item in classified
        ]
    )

for scenario in scenarios:
    scenario_name = scenario["scenario_name"]
    display(Markdown(f"## Scenario: `{scenario_name}`"))
    display(scenario_tables[scenario_name])
    display(Markdown("#### Deterministic classifier output (reference)"))
    display(classifier_tables[scenario_name])

In [ ]:
# Cell 4 - Compute scoring comparisons per scenario
comparison_results = []

for scenario in scenarios:
    scenario_name = scenario["scenario_name"]
    mocked_scores = scenario["mocked_quality_scores"]
    expected_stages = scenario["expected_stage_labels"]

    # Core comparison helper
    compared = compare_conversation(scenario)

    # Explicitly compute both methods for transparent reproducibility
    simple_score = simple_average_score(mocked_scores)
    weighted_score, _ = journey_weighted_score(mocked_scores, expected_stages)

    result = {
        "scenario_name": scenario_name,
        "simple_average_score": simple_score,
        "simple_rating": compared["simple_rating"],
        "journey_weighted_score": weighted_score,
        "journey_weighted_rating": compared["journey_weighted_rating"],
        "score_delta": compared["score_delta"],
        "critical_stage_failures": compared["critical_stage_failures"],
        "explanation": compared["explanation"],
    }
    comparison_results.append(result)

print("Computed scoring comparison for", len(comparison_results), "scenarios.")

In [ ]:
# Cell 5 - Summary table across all scenarios
summary_rows = []
for result in comparison_results:
    summary_rows.append({
        "scenario_name": result["scenario_name"],
        "simple_average_score": result["simple_average_score"],
        "simple_rating": result["simple_rating"],
        "journey_weighted_score": result["journey_weighted_score"],
        "journey_weighted_rating": result["journey_weighted_rating"],
        "score_delta": result["score_delta"],
        "rating_changed": result["simple_rating"] != result["journey_weighted_rating"],
        "critical_stage_failures": json.dumps(result["critical_stage_failures"]),
        "explanation": result["explanation"],
    })

summary_df = pd.DataFrame(summary_rows, columns=[
    "scenario_name",
    "simple_average_score",
    "simple_rating",
    "journey_weighted_score",
    "journey_weighted_rating",
    "score_delta",
    "rating_changed",
    "critical_stage_failures",
    "explanation",
])

display(Markdown("## Cross-scenario comparison summary"))
display(summary_df)

## Interpretation: Why Simple Average Can Mislead

Simple averaging treats all turns as equally important. In enterprise conversations, that can hide harmful behavior if low-risk turns are high quality while business-critical turns perform poorly.

Journey-aware verification keeps the same turn-level scores but changes aggregation so that failures in high-risk stages have greater impact on the final rating.

In [ ]:
# Cell 6 - Highlight acceptance scenarios
def _show_highlight(scenario_key):
    row = summary_df[summary_df["scenario_name"] == scenario_key]
    if row.empty:
        print(f"Scenario not found: {scenario_key}")
        return
    display(Markdown(f"### Highlight: `{scenario_key}`"))
    display(row)

    base_table = scenario_tables[scenario_key]
    critical_rows = base_table[base_table["expected_stage"].isin([
        "eligibility_or_suitability",
        "recommendation_or_guidance",
    ])]
    display(Markdown("#### Critical stage turns"))
    display(critical_rows)

_show_highlight("mortgage_hidden_eligibility_failure")
_show_highlight("investment_suitability_failure")

## Interpretation: Why Journey Stages Matter

The stages `eligibility_or_suitability` and `recommendation_or_guidance` are treated as critical because errors there can create direct compliance and customer-harm risk.

The weights in this notebook represent business importance, not universal human satisfaction. A conversation may feel fluent overall while still being unsafe or unsuitable at a critical decision point.

This supports the workshop theme of heterogeneous verification signals and long-horizon agent behavior: different parts of the journey contribute unequally to enterprise risk.

In [ ]:
# Cell 7 - Optional visual: simple vs journey-weighted score per scenario
display(Markdown("## Optional visual: simple average vs journey-aware weighted score"))
try:
    import matplotlib.pyplot as plt

    viz_df = summary_df[["scenario_name", "simple_average_score", "journey_weighted_score"]].copy()
    x = range(len(viz_df))
    width = 0.38

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar([i - width / 2 for i in x], viz_df["simple_average_score"], width=width, label="Simple Average")
    ax.bar([i + width / 2 for i in x], viz_df["journey_weighted_score"], width=width, label="Journey-Weighted")

    ax.set_xticks(list(x))
    ax.set_xticklabels(viz_df["scenario_name"], rotation=20, ha="right")
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel("Score (0 to 1)")
    ax.set_title("Simple vs Journey-Aware Aggregation")
    ax.axhline(0.75, linestyle="--", linewidth=1, color="green", label="PASS threshold")
    ax.axhline(0.60, linestyle="--", linewidth=1, color="orange", label="WARNING threshold")
    ax.legend()
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("Matplotlib unavailable or plotting failed. Skipping visual.")
    print(type(exc).__name__, str(exc))
    display(summary_df[["scenario_name", "simple_average_score", "journey_weighted_score"]])

In [ ]:
# Cell 8 - Acceptance checks and explicit downgrade evidence
acceptance = summary_df[summary_df["scenario_name"] == "mortgage_hidden_eligibility_failure"].iloc[0]
strong_case = summary_df[summary_df["scenario_name"] == "investment_suitability_failure"].iloc[0]

print("Acceptance scenario: mortgage_hidden_eligibility_failure")
print("simple_average_score:", acceptance["simple_average_score"], "simple_rating:", acceptance["simple_rating"])
print("journey_weighted_score:", acceptance["journey_weighted_score"], "journey_weighted_rating:", acceptance["journey_weighted_rating"])
print("critical_stage_failures:", acceptance["critical_stage_failures"])

print()
print("Secondary scenario: investment_suitability_failure")
print("simple_average_score:", strong_case["simple_average_score"], "simple_rating:", strong_case["simple_rating"])
print("journey_weighted_score:", strong_case["journey_weighted_score"], "journey_weighted_rating:", strong_case["journey_weighted_rating"])
print("critical_stage_failures:", strong_case["critical_stage_failures"])

assert acceptance["simple_rating"] == "PASS", "Expected simple PASS for acceptance scenario."
assert acceptance["journey_weighted_rating"] in {"WARNING", "FAIL"}, "Expected weighted WARNING or FAIL for acceptance scenario."

## Paper-Ready Takeaway

The journey-aware verifier does not replace existing conversational quality metrics. Instead, it changes the aggregation logic so that failures in business-critical journey stages are not diluted by high scores in low-risk stages.